In [ ]:
import pandas as pd
from google.colab import drive
pd.set_option('display.max_columns', None)

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df= pd.read_csv('/content/drive/My Drive/Medical_Insurance_Prediction_Project/medical_insurance_cleaned_engineered.csv')

In [ ]:
df

,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,dependents,bmi,smoker,alcohol_freq,visits_last_year,hospitalizations_last_3yrs,days_hospitalized_last_3yrs,medication_count,systolic_bp,diastolic_bp,ldl,hba1c,plan_type,network_tier,deductible,copay,policy_term_years,annual_medical_cost,annual_premium,hypertension,diabetes,asthma,copd,cardiovascular_disease,cancer_history,kidney_disease,liver_disease,arthritis,mental_health,had_major_procedure,bp_category,bmi_group,ldl_group,hba1c_group,annual_medical_cost_grouped
0,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,1,27.4,Never,Never,2,0,0,4,121.0,76.0,123.8,5.28,Preferred Provider Organization,Bronze,1000,20,4,6938.06,876.05,0,0,0,0,0,0,0,0,1,0,0,Elevated,Overweight,Near Optimal,Normal,Low
1,79,Female,North,Urban,12800.0,High School Dropout,Married,Employed,3,1,26.6,Never,Weekly,2,0,0,3,131.0,79.0,97.3,4.82,Point-of-Service,Gold,1000,10,1,1632.61,445.10,0,0,0,0,0,0,0,0,1,1,0,Hypertension Stage 1,Overweight,Optimal,Normal,Very Low
2,68,Male,North,Rural,40700.0,High School,Married,Retired,5,3,31.5,Never,Never,1,0,0,4,160.0,84.0,129.5,5.51,Health Maintenance Organization,Platinum,500,20,10,7661.01,1538.02,1,0,0,0,0,1,0,0,0,1,0,Hypertension Stage 1,Obese I,Near Optimal,Normal,Low
3,15,Male,North,Suburban,15600.0,College,Married,Self-employed,5,3,31.6,Never,Never,0,0,0,1,104.0,68.0,160.3,8.50,Health Maintenance Organization,Silver,500,20,5,5130.27,820.63,0,1,0,0,0,0,0,0,0,0,0,Normal,Obese I,High,Diabetes,Low
4,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,0,30.5,Never,Daily,3,0,0,2,136.0,83.0,171.0,5.20,Point-of-Service,Platinum,500,10,7,1700.73,500.93,1,0,0,0,0,0,0,0,1,0,0,Hypertension Stage 1,Obese I,High,Normal,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,50,Male,West,Urban,127200.0,High School Dropout,Married,Employed,2,0,28.2,Never,Occasional,0,0,0,1,115.0,74.0,102.5,4.98,Preferred Provider Organization,Bronze,500,10,10,1295.04,329.32,0,0,0,0,0,0,0,0,0,0,0,Normal,Overweight,Near Optimal,Normal,Very Low
99996,42,Male,East,Suburban,21600.0,High School,Married,Employed,2,0,27.6,Never,Occasional,0,0,0,1,101.0,66.0,177.1,5.66,Preferred Provider Organization,Silver,5000,20,4,1451.73,424.21,0,0,0,0,0,0,0,0,0,0,0,Normal,Overweight,High,Normal,Very Low
99997,41,Male,West,Rural,81900.0,High School,Divorced,Unemployed,1,0,29.8,Former,Weekly,7,0,0,1,128.0,83.0,118.8,5.52,Preferred Provider Organization,Gold,500,30,9,2291.00,534.90,1,0,0,0,0,0,0,0,0,0,0,Hypertension Stage 1,Overweight,Near Optimal,Normal,Low
99998,51,Female,South,Urban,43400.0,Doctorate,Single,Unemployed,3,2,21.9,Never,Occasional,4,0,0,2,110.0,73.0,134.9,5.25,Point-of-Service,Bronze,2000,20,3,1279.76,342.86,0,0,0,0,0,0,0,0,0,1,0,Normal,Normal,Borderline High,Normal,Very Low


In [ ]:
columns_to_drop = ['bp_category', 'bmi_group', 'ldl_group', 'hba1c_group', 'annual_medical_cost_grouped']
df = df.drop(columns=columns_to_drop)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Define target variable
target_variable = 'annual_premium'

# Separate features (X) and target (y)
X = df.drop(columns=[target_variable])
y = df[target_variable]

# Identify categorical and numerical features from X
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X.select_dtypes(include=['number']).columns.tolist()

# Preprocessor
preprocessor = ColumnTransformer(transformers=[('num', 'passthrough', numerical_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# CPU-SAFE XGBoost
xgb_regressor = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, tree_method="hist", predictor="cpu_predictor")

# Pipeline
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', xgb_regressor)])

In [27]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 38 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   age                          100000 non-null  int64  
 1   sex                          100000 non-null  object 
 2   region                       100000 non-null  object 
 3   urban_rural                  100000 non-null  object 
 4   income                       100000 non-null  float64
 5   education                    100000 non-null  object 
 6   marital_status               100000 non-null  object 
 7   employment_status            100000 non-null  object 
 8   household_size               100000 non-null  int64  
 9   dependents                   100000 non-null  int64  
 10  bmi                          100000 non-null  float64
 11  smoker                       100000 non-null  object 
 12  alcohol_freq                 100000 non-null  object 
 13  

In [26]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 100000 entries, 0 to 99999
Series name: annual_premium
Non-Null Count   Dtype  
--------------   -----  
100000 non-null  float64
dtypes: float64(1)
memory usage: 781.4 KB


In [ ]:
from sklearn.model_selection import train_test_split
import joblib

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:59:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age', 'income',
                                                   'household_size',
                                                   'dependents', 'bmi',
                                                   'visits_last_year',
                                                   'hospitalizations_last_3yrs',
                                                   'days_hospitalized_last_3yrs',
                                                   'medication_count',
                                                   'systolic_bp',
                                                   'diastolic_bp', 'ldl',
                                                   'hba1c', 'deductible',
                                                   'copay', 'policy_term_years',
                                                   'annual_medical_cost',
                                                   'hypertens...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=100, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Absolute Error (MAE): 6.21
Mean Squared Error (MSE): 5878.92
Root Mean Squared Error (RMSE): 76.67
R-squared (R2): 0.96


In [ ]:
import joblib
joblib.dump(model,"/content/drive/MyDrive/Medical_Insurance_Prediction_Project/xgboost_model_cpu.pkl")

['/content/drive/MyDrive/Medical_Insurance_Prediction_Project/xgboost_model_cpu.pkl']

In [28]:
import xgboost as xgb
print(f"XGBoost version in Colab: {xgb.__version__}")

XGBoost version in Colab: 3.2.0
